# Estudio 06 · Matriz de clima

Preregistro `CLIMATE_MATRIX` (commit `505b9b6`; respuestas de Rosa y filtro de la validación dejando una temporada
fuera en `dc1a709`; código de e1, e2 y n1 commiteado antes de correr). Fijado DESPUÉS de ver la temporada 2026.

12 variantes sobre L + 5a: medias (b1, b2, b3, b4), ventanas (d1 a d5), anomalías (e1, e2) y Niño monótono (n1).
Cada una frente a L + 5a con `SEASON_2026` (el Brier sobre 2010, 2015, 2016, 2022, 2023 y 2026). Se adopta si además
mejora el Brier en >= 16 de las 24 celdas de la validación dejando una temporada fuera. Cruces entre adoptadas
(`select_matrix_base`) y d1, d2 y e1 frente a b3. Entre 15 y 81 comparaciones: crece la probabilidad de un «gana» por
azar.

In [1]:
import json
import sys
import unittest

import numpy as np
import pandas as pd

from denguekay.config.paths import GOLD, MODELS, RESEARCH
from denguekay.evaluation import episodes as diag
from denguekay.evaluation import protocol as pa
from denguekay.model import horizon_strategies as eh
from denguekay.model import probabilistic_output as sp
from denguekay.model import tracking
from denguekay.model.device import xgboost_settings

STUDY = RESEARCH / "06_model_improvement"
sys.path.insert(0, str(STUDY / "code"))
import phase1b_prime as pb
import phase3_onsets as p3
import phase4_enso as p4
import phase5_counts as p5
import phase6_other_families as p6
import phase_2026 as p26
import phase_b3_full_climate as b3
import phase_b_rain as pbr
import phase_climate_matrix as cm
import phase_d_weather_windows as pdw
import preregistration as pr

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 400)
METRICS = STUDY / "docs" / "metricas" / "matriz_clima.json"
OUT = MODELS / "experimentos" / "mejora_modelo" / "matriz_clima"
OUT.mkdir(parents=True, exist_ok=True)
SEED = pr.SEEDS[0]
device = xgboost_settings()
inp = p26.inputs_2026()
origins, sources = p26.add_climate(inp)
builder = cm.E1Builder(sources)
base_cols, hp = p3._config()
COLS = {"L + 5a": list(base_cols), **{k: [*base_cols, *v] for k, v in pr.MATRIX_VARIANTS.items()}}
HP = {k: (cm.monotone_hp(hp, c, pr.MATRIX_MONOTONE[k]) if k in pr.MATRIX_MONOTONE else hp) for k, c in COLS.items()}
PREP = {k: (builder.prepare_block() if k in ("e1", "e2") else None) for k in COLS}
BLOCKS = p26.blocks()
CUTOFFS = {b: eh._common_cutoffs(inp["matrices"], kind, per)[1] for b, kind, per in BLOCKS}
print({k: len(v) for k, v in COLS.items()}, HP["n1"]["monotone_constraints"][-1])
device

{'L + 5a': 9, 'b1': 10, 'b2': 11, 'b3': 13, 'b4': 12, 'd1': 23, 'd2': 24, 'd3': 13, 'd4': 15, 'd5': 13, 'e1': 23, 'e2': 24, 'n1': 13} 1


{'device': 'cuda', 'n_jobs': 12}

## 1 · Guardas de datos y de columnas nuevas (antes de entrenar)

In [2]:
gold = pd.read_csv(GOLD / "meteo_socio_epi_piura_2017_2025_h2_gold.csv", dtype={"ubigeo": str},
                   float_precision="round_trip")


def run_tests(names):
    suite = unittest.TestSuite(unittest.defaultTestLoader.loadTestsFromName(n) for n in names)
    result = unittest.TextTestRunner(verbosity=0).run(suite)
    return {"tests": names, "corridos": result.testsRun, "fallas": len(result.failures) + len(result.errors),
            "pasa": result.wasSuccessful()}


sys.path.insert(0, str(STUDY / "tests"))
climate_cols = sorted({c for k, v in pr.MATRIX_VARIANTS.items() if k not in ("e1", "e2") for c in v})
used = pd.concat([inp["matrices"][h] for h in p3.H]).origen_cierre.max()
guards = {
    "G40": p26.guard_series(inp),
    "G17": pb.guard_g17(gold, inp["panel"]),
    "G25": pbr.guard_definition(gold, inp["panel"]),
    "G30": b3.guard_temperature_definition(inp["panel"]),
    "G35a": pdw.guard_daily_matches_silver(sources["semanal_diario"], sources["chirps"]),
    "G35b": pdw.guard_consistency(origins),
    "G42": p26.guard_coverage(origins, climate_cols, used),
    "G43_lluvia_media5": pbr.guard_availability(origins, pr.STEP_B_REFERENCE_DAYS),
    "G43_ventanas": pdw.guard_availability(origins),
    "G43_temperatura": b3.guard_temperature_availability(origins),
    "G43_nino": p4.guard_g21(origins[origins.nino12_anom_ultimo.notna()]),
    "G44": run_tests(["test_phase4_enso.TestAvailability.test_no_leakage_by_perturbation",
                      "test_phase_b_rain.TestDefinition.test_no_leakage_by_perturbation",
                      "test_phase_b3_full_climate.TestTemperature.test_no_leakage_by_perturbation",
                      "test_phase_d_weather_windows.TestWindows.test_no_leakage_by_perturbation"]),
    "G45_temporada_2008": cm.guard_e1_definition(builder, origins, CUTOFFS["temporada_2008"]),
    "G45_temporada_2026": cm.guard_e1_definition(builder, origins, CUTOFFS["temporada_2026"]),
    "G45_sintetica": run_tests(["test_phase_climate_matrix.TestE1.test_definition_anomaly_windows"]),
    "G46": run_tests(["test_phase_climate_matrix.TestE1.test_no_leakage_by_the_origin"]),
    "G47_temporada_2016": cm.guard_e1_cutoff(sources, origins, CUTOFFS["temporada_2016"]),
    "G47_temporada_2026": cm.guard_e1_cutoff(sources, origins, CUTOFFS["temporada_2026"]),
    "G48": cm.guard_e1_coverage(builder, origins, list(CUTOFFS.values()), used),
    "G49_G50_sinteticas": run_tests(["test_phase_climate_matrix.TestN1.test_constraint_is_monotone_and_zero_constraint_is_neutral"]),
}
failed = [k for k, g in guards.items() if not g["pasa"]]
if failed:
    raise RuntimeError(f"Fallan guardas {failed}: se detiene la matriz. {guards}")
{k: g["pasa"] for k, g in guards.items()}

----------------------------------------------------------------------
Ran 4 tests in 0.233s

OK


----------------------------------------------------------------------
Ran 1 test in 0.253s

OK


----------------------------------------------------------------------
Ran 1 test in 0.221s

OK


----------------------------------------------------------------------
Ran 1 test in 0.225s

OK


{'G40': True,
 'G17': True,
 'G25': True,
 'G30': True,
 'G35a': True,
 'G35b': True,
 'G42': True,
 'G43_lluvia_media5': True,
 'G43_ventanas': True,
 'G43_temperatura': True,
 'G43_nino': True,
 'G44': True,
 'G45_temporada_2008': True,
 'G45_temporada_2026': True,
 'G45_sintetica': True,
 'G46': True,
 'G47_temporada_2016': True,
 'G47_temporada_2026': True,
 'G48': True,
 'G49_G50_sinteticas': True}

## 2 · Reutilización (G41), entrenamiento y guardas de n1 (G49, G50)

In [3]:
T26 = MODELS / "experimentos" / "mejora_modelo" / "temporada_2026"
PB = MODELS / "experimentos" / "mejora_modelo" / "paso_b"
tag = lambda k: k.replace(" + ", "_").replace(" ", "")
variants, reuse_log = {}, {}
for k in COLS:
    path = OUT / f"predicciones_{tag(k)}.csv.gz"
    if path.exists():   # an earlier run of this notebook (same code and data)
        variants[k] = p26.evaluate_saved(path, inp["serie"])
        reuse_log[k] = "corrida anterior de este notebook"
        continue
    if k in pr.MATRIX_REUSE:
        saved = p26.read_saved(T26 / f"predicciones_{tag(k)}.csv.gz")
        rerun = p3.predict_blocks_weighted(inp["matrices"], origins, BLOCKS, SEED, COLS[k], HP[k])
        g = p26.guard_old_blocks({SEED: rerun}, {SEED: saved[SEED]})
        guards[f"G41_{k}"] = g
        if g["pasa"]:
            variants[k] = p26.evaluate_saved(T26 / f"predicciones_{tag(k)}.csv.gz", inp["serie"])
            reuse_log[k] = "temporada_2026 (G41 con la semilla 17)"
            continue
        reuse_log[k] = "G41 falló: reentrenada completa"
    preds, evals = {}, {}
    for s in pr.SEEDS:
        pred = p3.predict_blocks_weighted(inp["matrices"], origins, BLOCKS, s, COLS[k], HP[k], prepare_block=PREP[k])
        pred.attrs.pop("ajustes")
        preds[s] = pred
        evals[s] = p5.evaluate_mixture(pred, inp["serie"])
    variants[k] = {"predicciones": preds, "evaluaciones": evals}
    pd.concat([p.assign(semilla=s) for s, p in preds.items()], ignore_index=True).to_csv(path, index=False)
    reuse_log.setdefault(k, "entrenada")
    if k in ("b1", "b2"):
        guards[f"G41_{k}"] = p26.guard_old_blocks(preds, p26.read_saved(PB / f"predicciones_{k}.csv.gz"))
print(reuse_log)
{k: g for k, g in guards.items() if k.startswith("G41")}

{'L + 5a': 'temporada_2026 (G41 con la semilla 17)', 'b1': 'entrenada', 'b2': 'entrenada', 'b3': 'temporada_2026 (G41 con la semilla 17)', 'b4': 'entrenada', 'd1': 'temporada_2026 (G41 con la semilla 17)', 'd2': 'temporada_2026 (G41 con la semilla 17)', 'd3': 'entrenada', 'd4': 'entrenada', 'd5': 'entrenada', 'e1': 'entrenada', 'e2': 'entrenada', 'n1': 'entrenada'}


{'G41_L + 5a': {'filas': 195000,
  'guardadas_sin_par': 0,
  'filas_nuevas_en_bloques_guardados': 0,
  'diferencia_maxima': {'casos_predichos': 0.0, 'probabilidad_brote': 0.0},
  'tolerancia': 1e-06,
  'pasa': True},
 'G41_b1': {'filas': 915525,
  'guardadas_sin_par': 0,
  'filas_nuevas_en_bloques_guardados': 975,
  'diferencia_maxima': {'casos_predichos': 0.0, 'probabilidad_brote': 0.0},
  'tolerancia': 1e-06,
  'pasa': True},
 'G41_b2': {'filas': 915525,
  'guardadas_sin_par': 0,
  'filas_nuevas_en_bloques_guardados': 975,
  'diferencia_maxima': {'casos_predichos': 0.0, 'probabilidad_brote': 0.0},
  'tolerancia': 1e-06,
  'pasa': True},
 'G41_b3': {'filas': 195000,
  'guardadas_sin_par': 0,
  'filas_nuevas_en_bloques_guardados': 0,
  'diferencia_maxima': {'casos_predichos': 0.0, 'probabilidad_brote': 0.0},
  'tolerancia': 1e-06,
  'pasa': True},
 'G41_d1': {'filas': 195000,
  'guardadas_sin_par': 0,
  'filas_nuevas_en_bloques_guardados': 0,
  'diferencia_maxima': {'casos_predichos': 

In [4]:
# G49: n1 monótono en el Niño con el modelo de cada bloque (semilla 17); G50: sin restricción, n1 = b3
g49 = {}
for b in BLOCKS:
    fit = cm.fit_block(inp, origins, b, SEED, COLS["n1"], HP["n1"])
    g49[b[0]] = cm.monotone_check(fit, COLS["n1"])
guards["G49"] = {"bloques": g49, "pasa": all(v["pasa"] for v in g49.values()),
                 "peor_diferencia": min(v["peor_diferencia"] for v in g49.values())}
zero = cm.monotone_hp(hp, COLS["n1"], {})
blk = [b for b in BLOCKS if b[0] == "temporada_2026"]
n1_zero = p3.predict_blocks_weighted(inp["matrices"], origins, blk, SEED, COLS["n1"], zero)
b3_saved = variants["b3"]["predicciones"][SEED]
guards["G50"] = p26.guard_old_blocks({SEED: b3_saved[b3_saved.bloque.eq("temporada_2026")]},
                                     {SEED: n1_zero.drop(columns=["estrategia", "formulacion", "semilla"], errors="ignore")})
# a failed G41 only means retraining (done above, reported); G49 and G50 stop the matrix
for k in ("G49", "G50"):
    if not guards[k]["pasa"]:
        raise RuntimeError(f"Falla {k}: se detiene la matriz. {guards[k]}")
{k: (guards[k]["pasa"], guards[k].get("diferencia_maxima", guards[k].get("peor_diferencia"))) for k in guards if k[:3] in ("G41", "G49", "G50")}

{'G49_G50_sinteticas': (True, None),
 'G41_L + 5a': (True, {'casos_predichos': 0.0, 'probabilidad_brote': 0.0}),
 'G41_b1': (True, {'casos_predichos': 0.0, 'probabilidad_brote': 0.0}),
 'G41_b2': (True, {'casos_predichos': 0.0, 'probabilidad_brote': 0.0}),
 'G41_b3': (True, {'casos_predichos': 0.0, 'probabilidad_brote': 0.0}),
 'G41_d1': (True, {'casos_predichos': 0.0, 'probabilidad_brote': 0.0}),
 'G41_d2': (True, {'casos_predichos': 0.0, 'probabilidad_brote': 0.0}),
 'G49': (True, 0.0),
 'G50': (True, {'casos_predichos': 0.0, 'probabilidad_brote': 0.0})}

## 3 · Validación dejando una temporada fuera (necesaria para el filtro de adopción)

In [5]:
loso = {}
for k in COLS:
    path = OUT / f"fuera_de_temporada_{tag(k)}.csv.gz"
    if path.exists():
        preds = p26.read_saved(path)
    else:
        preds = {s: cm.loso_predictions(inp, origins, COLS[k], HP[k], s, builder=builder if k in ("e1", "e2") else None)
                 for s in pr.SEEDS}
        pd.concat([p.assign(semilla=s) for s, p in preds.items()], ignore_index=True).to_csv(path, index=False)
    loso[k] = p26.loso_metrics(preds, inp["serie"])
loso_all = pd.concat(loso, names=["variante"]).reset_index(level=0)
cells_better = {k: cm.loso_better_cells(loso_all, k) for k in pr.MATRIX_VARIANTS}
cells_better

{'b1': 16,
 'b2': 15,
 'b3': 19,
 'b4': 20,
 'd1': 22,
 'd2': 23,
 'd3': 21,
 'd4': 12,
 'd5': 17,
 'e1': 17,
 'e2': 17,
 'n1': 19}

## 4 · Veredictos frente a L + 5a, adopción, cruces y d1, d2, e1 frente a b3

In [6]:
outbreaks = p26.outbreak_weeks(inp["serie"])
base = variants["L + 5a"]
results = {k: p26.compare_2026(variants[k], base, outbreaks) for k in pr.MATRIX_VARIANTS}
adopted = [k for k in pr.MATRIX_SIMPLICITY
           if pr.adopt_matrix(results[k]["brier"], results[k]["alerta"], results[k]["conteo"], cells_better[k])]
summary = pd.DataFrame([{"variante": k, "columnas": len(pr.MATRIX_VARIANTS[k]),
                         **{f"brier_h{h}": v for h, v in zip(p3.H, results[k]["brier"].veredicto)},
                         **{f"f1_h{h}": v for h, v in zip(p3.H, results[k]["alerta"].veredicto)},
                         **{f"conteo_h{h}": v for h, v in zip(p3.H, results[k]["conteo"].veredicto)},
                         "gana_season_2026": results[k]["adopta"], "celdas_loso_mejores": cells_better[k],
                         "adoptada": k in adopted} for k in pr.MATRIX_SIMPLICITY])
crosses = {}
for i, a in enumerate([k for k in pr.MATRIX_SIMPLICITY if k in adopted]):
    for b in [k for k in pr.MATRIX_SIMPLICITY if k in adopted][i + 1:]:
        crosses[f"{b} frente a {a}"] = p26.compare_2026(variants[b], variants[a], outbreaks)
beats = lambda a, b: crosses[f"{a} frente a {b}"]["adopta"]
winner = pr.select_matrix_base(adopted, beats)
fixed = {f"{a} frente a {b}": p26.compare_2026(variants[a], variants[b], outbreaks) for a, b in pr.MATRIX_FIXED_CROSSES}
print("Adoptadas:", adopted, "| base:", winner or "L + 5a", "| comparaciones:", pr.matrix_comparisons(len(adopted)))
summary

Adoptadas: ['b3'] | base: b3 | comparaciones: 15


,variante,columnas,brier_h2,brier_h3,brier_h4,f1_h2,f1_h3,f1_h4,conteo_h2,conteo_h3,conteo_h4,gana_season_2026,celdas_loso_mejores,adoptada
0,b1,1,empate,empate,empate,empate,empate,empate,empate,empate,empate,False,16,False
1,b2,2,empate,empate,empate,empate,empate,empate,empate,empate,empate,False,15,False
2,b4,3,empate,empate,empate,empate,empate,empate,empate,empate,empate,False,20,False
3,d3,4,gana,empate,empate,empate,empate,empate,empate,empate,empate,False,21,False
4,d5,4,empate,empate,empate,empate,empate,empate,empate,empate,empate,False,17,False
5,b3,4,gana,gana,empate,empate,empate,empate,empate,empate,empate,True,19,True
6,n1,4,empate,empate,empate,empate,empate,empate,empate,empate,empate,False,19,False
7,d4,6,empate,empate,empate,empate,empate,empate,empate,empate,empate,False,12,False
8,d1,14,empate,empate,empate,empate,empate,empate,empate,empate,empate,False,22,False
9,e1,14,empate,empate,empate,empate,empate,empate,empate,empate,empate,False,17,False


In [7]:
cross_table = pd.DataFrame([{"cruce": k, **{f"brier_h{h}": v for h, v in zip(p3.H, r["brier"].veredicto)},
                             **{f"f1_h{h}": v for h, v in zip(p3.H, r["alerta"].veredicto)},
                             **{f"conteo_h{h}": v for h, v in zip(p3.H, r["conteo"].veredicto)}, "gana": r["adopta"]}
                            for k, r in {**crosses, **fixed}.items()])
cross_table

,cruce,brier_h2,brier_h3,brier_h4,f1_h2,f1_h3,f1_h4,conteo_h2,conteo_h3,conteo_h4,gana
0,d1 frente a b3,empate,empate,empate,empate,empate,empate,empate,empate,empate,False
1,d2 frente a b3,empate,empate,empate,empate,empate,empate,empate,empate,empate,False
2,e1 frente a b3,empate,empate,pierde,empate,pierde,empate,empate,pierde,empate,False


## 5 · Métricas por celda, HU0007-4, pendiente de Platt de n1 y SHAP

In [8]:
cells = pd.concat({k: p26.cell_metrics(v, inp["serie"]) for k, v in variants.items()}, names=["variante"]).reset_index(level=0)
show = ["variante", "horizonte", "bloque", "positivos", "brier", "bss_climatologia", "auprc", "auprc_bajo_umbral",
        "f1_050", "razon_mae", "razon_rmse"]
med = {k: pa.seed_median([e["metricas"] for e in v["evaluaciones"].values()]) for k, v in variants.items()}
hu = pd.concat({k: sp.hu0007_4_compliance(m[m.bloque.isin(["temporada_2026", "posterior_2026"])]) for k, m in med.items()},
               names=["variante"]).reset_index(level=0)
slopes = cm.platt_slopes(variants["n1"]["predicciones"][SEED])
print("Pendiente de Platt de n1: mínima", round(slopes.pendiente.min(), 3), "| bloques con pendiente <= 0:", int((slopes.pendiente <= 0).sum()))
cells[cells.bloque.isin(["temporada_2026", "posterior_2026"])][show].sort_values(["bloque", "horizonte", "variante"]).round(3)

Pendiente de Platt de n1: mínima 0.842 | bloques con pendiente <= 0: 0


,variante,horizonte,bloque,positivos,brier,bss_climatologia,auprc,auprc_bajo_umbral,f1_050,razon_mae,razon_rmse
0,L + 5a,2,posterior_2026,167.0,0.095,0.695,0.933,0.382,0.804,1.122,1.141
0,b1,2,posterior_2026,167.0,0.097,0.691,0.933,0.376,0.793,1.187,1.207
0,b2,2,posterior_2026,167.0,0.097,0.690,0.932,0.373,0.793,1.188,1.206
0,b3,2,posterior_2026,167.0,0.091,0.709,0.933,0.383,0.813,1.126,1.145
0,b4,2,posterior_2026,167.0,0.091,0.708,0.933,0.388,0.809,1.130,1.148
0,d1,2,posterior_2026,167.0,0.090,0.711,0.933,0.379,0.817,1.068,1.070
0,d2,2,posterior_2026,167.0,0.090,0.713,0.934,0.380,0.817,1.076,1.080
0,d3,2,posterior_2026,167.0,0.087,0.721,0.935,0.399,0.845,1.012,1.004
0,d4,2,posterior_2026,167.0,0.098,0.687,0.932,0.349,0.789,1.176,1.191
0,d5,2,posterior_2026,167.0,0.095,0.695,0.932,0.348,0.799,1.159,1.182


In [9]:
# SHAP (semilla 17) de las adoptadas y de las 3 mejores por el Brier (mediana de la diferencia frente a L + 5a)
six = p26.block_names(pr.SEASON_2026_DECIDING)
ref = med["L + 5a"].set_index(["horizonte", "bloque"]).brier
def brier_gap(k):
    d = med[k].set_index(["horizonte", "bloque"]).brier - ref
    return float(d[d.index.get_level_values("bloque").isin(six)].median())


brier_score = {k: brier_gap(k) for k in pr.MATRIX_VARIANTS}
top = sorted(brier_score, key=brier_score.get)[:pr.MATRIX_SHAP_TOP_BRIER]
shap_for = list(dict.fromkeys([*adopted, *top]))
shap = {}
for k in shap_for:
    for name in pr.MATRIX_SHAP_BLOCKS:
        b = next(x for x in BLOCKS if x[0] == name)
        fit = cm.fit_block(inp, origins, b, SEED, COLS[k], HP[k], PREP[k])
        shap[(k, name)] = sp.shap_importance_by_horizon(fit["regresor"], fit["x_reg"])
print("Mejores por el Brier:", {k: round(brier_score[k], 5) for k in top}, "| SHAP de:", shap_for)
pd.concat({f"{k} · {b}": t for (k, b), t in shap.items()}, names=["variante_bloque", "horizonte"]).round(4).T

Mejores por el Brier: {'b3': -0.00065, 'n1': -0.00063, 'd2': -0.00062} | SHAP de: ['b3', 'n1', 'd2']


variante_bloque                     b3 · temporada_2022                 b3 · temporada_2023                 b3 · temporada_2026                 n1 · temporada_2022                 n1 · temporada_2023                 n1 · temporada_2026          \
horizonte                                           h=2     h=3     h=4                 h=2     h=3     h=4                 h=2     h=3     h=4                 h=2     h=3     h=4                 h=2     h=3     h=4                 h=2     h=3   
hist_lag_a                                       0.0370  0.0734  0.0997              0.0792  0.1245  0.1638              0.0770  0.1146  0.1444              0.0399  0.0824  0.1172              0.0791  0.1227  0.1647              0.0963  0.1321   
hist_lag_b                                       0.0099  0.0254  0.0386              0.0278  0.0191  0.0156              0.0305  0.0199  0.0215              0.0066  0.0229  0.0393              0.0186  0.0121  0.0194              0.0267  0.0195   
hist_media_4                                     0.2473  0.2127  0.1753              0.2909  0.2546  0.2181              0.3263  0.2841  0.2422              0.2485  0.2186  0.1878              0.2911  0.2522  0.2175              0.3298  0.2877   
hist_semanas_pos_4                               0.0150  0.0131  0.0112              0.0133  0.0116  0.0098              0.0069  0.0069  0.0071              0.0232  0.0204  0.0180              0.0134  0.0117  0.0097              0.0195  0.0189   
semana_epi_seno                                  0.1648  0.2030  0.2255              0.1450  0.1769  0.1963              0.1395  0.1690  0.1918              0.1663  0.2050  0.2274              0.1667  0.2012  0.2224              0.1461  0.1755   
semana_epi_coseno                                0.0693  0.1179  0.1705              0.0795  0.1317  0.1859              0.0739  0.1267  0.1781              0.0724  0.1228  0.1767              0.0803  0.1325  0.1862              0.0754  0.1286   
propia_semanas_desde_ultimo_caso                 1.4895  1.3827  1.2799              1.4472  1.3427  1.2522              1.2551  1.1614  1.0809              1.4718  1.3681  1.2674              1.4671  1.3593  1.2667              1.2556  1.1587   
propia_pendiente_4                               0.0094  0.0082  0.0070              0.0016  0.0019  0.0020              0.0225  0.0238  0.0237              0.0092  0.0076  0.0053              0.0029  0.0032  0.0033              0.0232  0.0244   
propia_log_casos_temporada_anterior              0.0525  0.0636  0.0724              0.0513  0.0658  0.0765              0.0713  0.0817  0.0910              0.0566  0.0673  0.0761              0.0481  0.0617  0.0721              0.0631  0.0740   
hum_rel_media_media5_h2                          0.0259  0.0391  0.0450              0.0295  0.0426  0.0492              0.0418  0.0575  0.0661              0.0269  0.0388  0.0447              0.0287  0.0410  0.0480              0.0430  0.0572   
precip_total_mm_media5_h2                        0.0767  0.0926  0.0993              0.0609  0.0761  0.0823              0.0936  0.1222  0.1435              0.0687  0.0852  0.0924              0.0588  0.0729  0.0782              0.0926  0.1218   
temp_media_media5_h2                             0.3395  0.4030  0.4566              0.3344  0.3945  0.4470              0.3013  0.3540  0.3969              0.3380  0.3988  0.4510              0.3245  0.3835  0.4361              0.3208  0.3727   
nino12_anom_media3                               0.0431  0.0545  0.0630              0.0200  0.0261  0.0331              0.0066  0.0123  0.0240              0.0440  0.0555  0.0639              0.0188  0.0246  0.0299              0.0041  0.0054   
precip_total_mm_s0                                  NaN     NaN     NaN                 NaN     NaN     NaN                 NaN     NaN     NaN                 NaN     NaN     NaN                 NaN     NaN     NaN                 NaN     NaN   
precip_total_mm_media_s1_4                      

## 6 · Salidas

In [10]:
to_json = lambda d: {k: diag.to_json(x) for k, x in d.items() if isinstance(x, pd.DataFrame)}
content = {
    "dispositivo": device, "guardas": guards, "reutilizacion": reuse_log, "semanas_brote_2026": outbreaks,
    "resumen": diag.to_json(summary), "veredictos": {k: {**to_json(r), "adopta": r["adopta"]} for k, r in results.items()},
    "celdas_loso_mejores": cells_better, "adoptadas": adopted, "base_resultante": winner or "L + 5a",
    "numero_de_comparaciones": pr.matrix_comparisons(len(adopted)),
    "cruces": diag.to_json(cross_table), "metricas_por_celda": diag.to_json(cells), "hu0007_4": diag.to_json(hu),
    "pendiente_platt_n1": diag.to_json(slopes), "brier_mediano_frente_a_L_5a": brier_score,
    "shap_semilla_17": {f"{k} · {b}": diag.to_json(t.reset_index(names="horizonte")) for (k, b), t in shap.items()},
    "dejando_una_temporada_fuera": diag.to_json(loso_all),
}
content = json.loads(json.dumps(content, default=lambda o: o.item() if hasattr(o, "item") else str(o)))
for section, value in content.items():
    diag.save_metrics_section(METRICS, section, value)
verdicts = summary[["variante", "brier_h2", "brier_h3", "brier_h4", "adoptada"]]
run_id = tracking.log_study(
    "mejora_modelo_matriz_clima", pd.DataFrame(columns=["numero", "estado"]),
    parent={"params": {"fase": "matriz_clima", "preregistro": "505b9b6 + dc1a709", "adoptadas": ", ".join(adopted) or "ninguna",
                       "base_resultante": content["base_resultante"],
                       **{f"xgb.{k}": str(x) for k, x in device.items()}},
            "metricas": {}},
    artifacts=[METRICS], tables={"resumen": verdicts},
    description="Estudio 06: matriz de 12 variantes de clima sobre L + 5a con la regla SEASON_2026 y el filtro LOSO.")
run_id

C:\Users\Rosa\RufoEsEternoElRegreso\Denguekay\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


MLflow: estudio 3e33511d09124833a0d7fa63e875daa3 en 'denguekay/mejora_modelo_matriz_clima' (sqlite:///C:/Users/Rosa/RufoEsEternoElRegreso/Denguekay/mlflow.db)


'3e33511d09124833a0d7fa63e875daa3'